## Build Environmental Camera Covariates

Extract environmental predictors for each Snapshot USA camera used in the species-level analysis.

- **NLCD land-cover data** within each 1-km camera footprint.
- **TIGER primary roads** to calculate distance to the nearest major road.


In [59]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.mask import mask
from shapely.geometry import Point

In [51]:
# --------------------------------------------------
# Define file paths
# --------------------------------------------------

DATA_PATH = "../../data"
PREPROCESSED_PATH = "../preprocessed_data"
OUTPUT_PATH = "../../outputs/species_level_analysis"

# NLCD 2024 land-cover raster
NLCD_FILE = (
    f"{DATA_PATH}/nlcd/Annual_NLCD_LndCov_2024_CU_C1V1/"
    "Annual_NLCD_LndCov_2024_CU_C1V1.tif"
)

# TIGER 2025 primary roads
PRIMARY_ROADS_FILE = (
    f"{DATA_PATH}/tiger_roads/tl_2025_us_primaryroads/"
    "tl_2025_us_primaryroads.shp"
)

# 1-km camera footprints
CAMERA_FOOTPRINTS_FILE = (
    f"{PREPROCESSED_PATH}/ssusa_camera_footprints_1km.geojson"
)

# Camera-level site covariates from previous notebook
CAMERA_COVARIATES_FILE = (
    f"{OUTPUT_PATH}/camera_covariates.csv"
)

In [53]:
# Load camera-level covariates
# --------------------------------------------------

camera_covariates = pd.read_csv(
    CAMERA_COVARIATES_FILE
)

print(
    f"Camera covariates: {camera_covariates.shape}"
)


Camera covariates: (7340, 6)


### Extract NLCD Land-Cover Percentages

Load the 1-km camera footprints and the NLCD 2024 land-cover raster.

In [54]:
camera_buffers = gpd.read_file(
    CAMERA_FOOTPRINTS_FILE
)

nlcd = rasterio.open(
    NLCD_FILE
)

# Inspect spatial datasets
print(f"Camera footprints: {camera_buffers.shape}")
print(f"Camera CRS: {camera_buffers.crs}")
print(f"NLCD CRS: {nlcd.crs}")


Camera footprints: (7340, 4)
Camera CRS: EPSG:5070
NLCD CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


ERROR 1: PROJ: internal_proj_identify: /Users/neelima/miniconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


In [56]:
# Align camera footprints with NLCD CRS
# --------------------------------------------------

if camera_buffers.crs != nlcd.crs:
    camera_buffers = camera_buffers.to_crs(
        nlcd.crs
    )

print(
    "Camera CRS after alignment:",
    camera_buffers.crs
)

Camera CRS after alignment: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


### Extract NLCD Percentages Within a Camera Footprint


In [57]:
# Define NLCD land-cover groups
# --------------------------------------------------

NLCD_GROUPS = {
    "forest_pct": [41, 42, 43],
    "grassland_pct": [71, 72, 73, 74],
    "wetland_pct": [90, 95],
    "developed_pct": [21, 22, 23, 24],
    "cropland_pct": [81, 82],
}

In [60]:
# --------------------------------------------------
# Extract NLCD percentages for one camera footprint
# --------------------------------------------------

def extract_nlcd_percentages(geometry, raster, class_groups):
    """
    Calculate NLCD land-cover percentages within one camera footprint.
    """

    # Clip the NLCD raster to the camera footprint
    try:
        clipped, _ = mask(
            raster,
            [geometry],
            crop=True,
            filled=False,
        )

    # Return missing values if the footprint does not overlap the raster
    except ValueError:
        return {
            column: np.nan
            for column in class_groups
        }

    # Extract the first raster band
    pixels = clipped[0]

    # Keep only valid, unmasked pixels
    valid_pixels = pixels.compressed()

    # Remove NoData values if defined
    if raster.nodata is not None:
        valid_pixels = valid_pixels[
            valid_pixels != raster.nodata
        ]

    # Return missing values if no valid NLCD pixels remain
    if len(valid_pixels) == 0:
        return {
            column: np.nan
            for column in class_groups
        }

    results = {}

    # Calculate percentage of pixels in each land-cover group
    for column, nlcd_codes in class_groups.items():

        class_count = np.isin(
            valid_pixels,
            nlcd_codes,
        ).sum()

        results[column] = (
            class_count / len(valid_pixels)
        ) * 100

    # Record number of valid NLCD pixels used
    results["valid_nlcd_pixels"] = len(valid_pixels)

    return results

#### Identify Cameras within NLCD range 

NLCD file covers the contiguous United States
Cameras outside the NLCD coverage cannot receive NLCD land-cover covariates and will be identified before extraction.

In [61]:
from shapely.geometry import box

# Create a polygon representing the NLCD raster extent
nlcd_extent = box(
    nlcd.bounds.left,
    nlcd.bounds.bottom,
    nlcd.bounds.right,
    nlcd.bounds.top,
)

# Check whether each camera footprint overlaps the NLCD extent
camera_buffers["within_nlcd_extent"] = (
    camera_buffers.geometry.intersects(nlcd_extent)
)

print(
    "Cameras within NLCD extent:",
    camera_buffers["within_nlcd_extent"].sum(),
)

print(
    "Cameras outside NLCD extent:",
    (~camera_buffers["within_nlcd_extent"]).sum(),
)

Cameras within NLCD extent: 7263
Cameras outside NLCD extent: 77


In [62]:
# Keep cameras covered by the NLCD raster
# --------------------------------------------------

camera_buffers = (
    camera_buffers[
        camera_buffers["within_nlcd_extent"]
    ]
    .copy()
)

print(
    f"Remaining cameras: {len(camera_buffers):,}"
)

Remaining cameras: 7,263


In [63]:
# Align camera covariates with NLCD-covered cameras
# --------------------------------------------------

camera_covariates = (
    camera_covariates[
        camera_covariates["camera_fp_id"].isin(
            camera_buffers["camera_fp_id"]
        )
    ]
    .copy()
)

print(
    f"Camera covariates: {len(camera_covariates):,}"
)

Camera covariates: 7,263


#### Extract NLCD Land-Cover Percentages for All Cameras

In [65]:
from tqdm.auto import tqdm

tqdm.pandas()

# Apply the NLCD extraction function to the geometry
# of every retained camera footprint.
nlcd_results = camera_buffers["geometry"].progress_apply(
    lambda geometry: extract_nlcd_percentages(
        geometry,
        nlcd,
        NLCD_GROUPS,
    )
)

# Convert extracted dictionaries to a DataFrame
nlcd_covariates = pd.DataFrame(
    nlcd_results.tolist()
)

# Add camera identifier
nlcd_covariates.insert(
    0,
    "camera_fp_id",
    camera_buffers["camera_fp_id"].values,
)

print(
    "NLCD covariate table shape:",
    nlcd_covariates.shape
)

nlcd_covariates.head()

  0%|          | 0/7263 [00:00<?, ?it/s]

NLCD covariate table shape: (7263, 7)


,camera_fp_id,forest_pct,grassland_pct,wetland_pct,developed_pct,cropland_pct,valid_nlcd_pixels
0,-85.48810000_32.66049000,48.937392,0.516944,2.096496,34.348076,6.720276,3482
1,-85.48588000_32.66487000,47.230990,1.692970,1.463415,43.558106,3.672884,3485
2,-85.48485000_32.66440000,44.788975,1.492966,1.263279,47.516509,3.301751,3483
3,-85.48567000_32.66379000,45.047373,1.464255,1.464255,45.564169,3.904680,3483
4,-85.48630000_32.66455000,47.398513,1.715266,1.457976,42.510006,3.887936,3498


In [66]:
print(nlcd.res)

(30.0, 30.0)


In [67]:
# Validate Extracted NLCD Covariates
print("Number of cameras:", len(nlcd_covariates))
print("Unique camera IDs:", nlcd_covariates["camera_fp_id"].nunique())

print("\nMissing values:")
print(nlcd_covariates.isna().sum())

print("\nSummary statistics:")
display(
    nlcd_covariates.drop(columns="camera_fp_id").describe().round(2)
)

Number of cameras: 7263
Unique camera IDs: 7263

Missing values:
camera_fp_id         0
forest_pct           0
grassland_pct        0
wetland_pct          0
developed_pct        0
cropland_pct         0
valid_nlcd_pixels    0
dtype: int64

Summary statistics:


,forest_pct,grassland_pct,wetland_pct,developed_pct,cropland_pct,valid_nlcd_pixels
count,7263.00,7263.00,7263.00,7263.00,7263.00,7263.00
mean,43.25,6.79,8.78,18.93,9.10,3484.73
std,34.64,18.87,17.22,28.31,17.43,14.80
min,0.00,0.00,0.00,0.00,0.00,2637.00
25%,5.85,0.00,0.00,1.55,0.00,3482.00
50%,42.21,0.20,1.21,4.82,0.29,3485.00
75%,76.09,2.21,8.55,21.81,9.76,3488.00
max,100.00,100.00,100.00,100.00,95.26,3501.00


In [68]:
# Save NLCD Covariates
NLCD_OUTPUT_FILE = f"{OUTPUT_PATH}/camera_nlcd_covariates.csv"

nlcd_covariates.to_csv(
    NLCD_OUTPUT_FILE,
    index=False,
)

print(f"Saved {len(nlcd_covariates):,} camera records to:")
print(NLCD_OUTPUT_FILE)

Saved 7,263 camera records to:
../../outputs/species_level_analysis/camera_nlcd_covariates.csv


### Calculate Distance to Primary Roads

Calculate the distance from each camera location to the nearest primary road.

Unlike the NLCD covariates, which summarize land cover within the 1-km camera footprint, road distance is measured from the camera's actual geographic location.

First, convert the camera latitude and longitude coordinates into point geometries.


In [75]:
# Create camera point geometries
# --------------------------------------------------

# Convert the camera-level DataFrame into a GeoDataFrame.
camera_points = gpd.GeoDataFrame(

    # Keep all existing camera-level covariates.
    camera_covariates.copy(),

    # Create a geographic point for each camera
    # using longitude as X and latitude as Y.
    geometry=gpd.points_from_xy(
        camera_covariates["Longitude"],
        camera_covariates["Latitude"],
    ),

    # Latitude and longitude are stored in WGS84 coordinates.
    crs="EPSG:4326",
)


# Confirm that one point was created for every retained camera.
print(
    "Camera point rows:",
    len(camera_points)
)


# Confirm the coordinate reference system.
print(
    "Camera point CRS:",
    camera_points.crs
)


# Preview the resulting GeoDataFrame.
camera_points.head()

Camera point rows: 7263
Camera point CRS: EPSG:4326


,camera_fp_id,Latitude,Longitude,Total_Survey_Nights,Habitat,Development_Level,geometry
8,-85.48810000_32.66049000,32.66049,-85.48810,25,Forest,Suburban,POINT (-85.4881 32.66049)
9,-85.48588000_32.66487000,32.66487,-85.48588,24,Forest,Suburban,POINT (-85.48588 32.66487)
10,-85.48485000_32.66440000,32.66440,-85.48485,20,Forest,Suburban,POINT (-85.48485 32.6644)
11,-85.48567000_32.66379000,32.66379,-85.48567,24,Forest,Suburban,POINT (-85.48567 32.66379)
12,-85.48630000_32.66455000,32.66455,-85.48630,24,Forest,Suburban,POINT (-85.4863 32.66455)


In [76]:
# Load TIGER primary roads
# --------------------------------------------------

# Read the primary-road line features into a GeoDataFrame.
primary_roads = gpd.read_file(
    PRIMARY_ROADS_FILE
)

# Check the number of road features loaded.
print(
    "Primary roads shape:",
    primary_roads.shape
)

# Check the coordinate reference system.
# The source road data use geographic coordinates,
# so they will be reprojected before calculating distances.
print(
    "Primary roads CRS:",
    primary_roads.crs
)

# Preview the road dataset.
primary_roads.head()

Primary roads shape: (17500, 5)
Primary roads CRS: EPSG:4269


,LINEARID,FULLNAME,RTTYP,MTFCC,geometry
0,1108296490085,Liberty Expy,M,S1100,"LINESTRING (-84.22256 31.62253, -84.22222 31.6..."
1,1108296492349,Liberty Expy,M,S1100,"LINESTRING (-106.426 31.82636, -106.42382 31.8..."
2,1108296492353,Liberty Expy,M,S1100,"LINESTRING (-106.42708 31.82613, -106.4238 31...."
3,1108296490069,Liberty Expy,M,S1100,"LINESTRING (-84.11222 31.56809, -84.11203 31.5..."
4,1108296312361,Liberty Expy,M,S1100,"LINESTRING (-84.22133 31.62254, -84.22187 31.6..."


In [71]:
# Reproject camera points to the NLCD projected CRS
camera_points_projected = camera_points.to_crs(nlcd.crs)

# Reproject primary roads to the same CRS
primary_roads_projected = primary_roads.to_crs(nlcd.crs)

print("Camera points CRS:")
print(camera_points_projected.crs)

print("\nPrimary roads CRS:")
print(primary_roads_projected.crs)

print(
    "\nCRS match:",
    camera_points_projected.crs == primary_roads_projected.crs,
)

Camera points CRS:
PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Primary roads CRS:
PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["fa

In [77]:
# Reproject cameras and roads to a common CRS
# --------------------------------------------------

# Reproject camera points from WGS84 latitude/longitude
# to the same projected CRS used by the NLCD raster.
camera_points_projected = camera_points.to_crs(
    nlcd.crs
)


# Reproject the primary-road line features
# to the exact same projected CRS.
primary_roads_projected = primary_roads.to_crs(
    nlcd.crs
)


# Confirm that both spatial datasets now use the same CRS.
print(
    "CRS match:",
    camera_points_projected.crs
    == primary_roads_projected.crs
)

CRS match: True


In [78]:
# Calculate distance to nearest primary road
# --------------------------------------------------

# Perform a nearest spatial join.
# For each camera point, GeoPandas finds the closest
# road geometry and calculates the distance to it.

camera_road_distance = gpd.sjoin_nearest(
    
    # Keep only the camera ID and point geometry.
    camera_points_projected[
        [
            "camera_fp_id",
            "geometry",
        ]
    ],

    # Only the road geometry is needed for the distance calculation.
    primary_roads_projected[
        [
            "geometry",
        ]
    ],

    # Keep every camera even if a nearest road is not found.
    how="left",

    # Store the calculated distance in meters in this new column.
    distance_col="distance_to_primary_road_m",
)


# Keep only the variables needed for the final road-distance table.
camera_road_distance = camera_road_distance[
    [
        "camera_fp_id",
        "distance_to_primary_road_m",
    ]
]


# A camera can occasionally match multiple equally-near road segments.
# Keep one record per camera so the final table remains camera-level.
camera_road_distance = (
    camera_road_distance
    .drop_duplicates(
        subset="camera_fp_id"
    )
    .reset_index(drop=True)
)


# Validate the result.
print(
    "Road-distance rows:",
    len(camera_road_distance)
)

print(
    "Unique cameras:",
    camera_road_distance["camera_fp_id"].nunique()
)

camera_road_distance.head()

Road-distance rows: 7263
Unique cameras: 7263


,camera_fp_id,distance_to_primary_road_m
0,-85.48810000_32.66049000,8052.639400
1,-85.48588000_32.66487000,8388.136279
2,-85.48485000_32.66440000,8296.946743
3,-85.48567000_32.66379000,8272.448181
4,-85.48630000_32.66455000,8374.842564


In [79]:
# Validate nearest-road distances
# --------------------------------------------------

# Check the overall shape of the road-distance table.
# Expected: 7,263 cameras × 2 columns.
print(
    "Road-distance table shape:",
    camera_road_distance.shape
)


# Confirm that each camera appears only once.
print(
    "Unique camera IDs:",
    camera_road_distance["camera_fp_id"].nunique()
)


# Check whether any camera failed to receive
# a nearest-road distance.
print(
    "Missing distances:",
    camera_road_distance[
        "distance_to_primary_road_m"
    ].isna().sum()
)


# Distances should never be negative.
# This provides a basic validity check.
print(
    "Negative distances:",
    (
        camera_road_distance[
            "distance_to_primary_road_m"
        ] < 0
    ).sum()
)


# Summarize the distribution of distances in meters.
print("\nDistance summary in meters:")

display(
    camera_road_distance[
        "distance_to_primary_road_m"
    ]
    .describe()
    .to_frame()
    .round(2)
)

Road-distance table shape: (7263, 2)
Unique camera IDs: 7263
Missing distances: 0
Negative distances: 0

Distance summary in meters:


,distance_to_primary_road_m
count,7263.00
mean,28461.26
std,41116.53
min,12.92
25%,3581.40
50%,12429.93
75%,34822.72
max,212776.44


### Save Distance-to-Road Covariates

Save the nearest-primary-road distance for each retained camera.


In [81]:
# Save camera-to-road distances
# --------------------------------------------------

# Define the output file path.
ROAD_DISTANCE_OUTPUT_FILE = (
    f"{OUTPUT_PATH}/camera_road_distance.csv"
)


# Save one row per camera containing:
# - camera_fp_id
# - distance_to_primary_road_m
camera_road_distance.to_csv(
    ROAD_DISTANCE_OUTPUT_FILE,
    index=False,
)

# and display the output location.
print(
    f"Saved {len(camera_road_distance):,} camera records to:"
)

print(
    ROAD_DISTANCE_OUTPUT_FILE
)

Saved 7,263 camera records to:
../../outputs/species_level_analysis/camera_road_distance.csv
